# Netflix Recommendation System — Task 6: Recommendation Generation

This notebook uses our final Recommender interface to display Top-10 recommendations for 5 different users with various activity levels. We verify their watch history, display explainable rationales, identify success/failure cases, and list similar movies.

In [1]:
import sys
import os
import pandas as pd

sys.path.append(os.path.abspath('..'))
from src import config
from src.recommender import Recommender

## 1. Load Recommender

In [2]:
rec = Recommender(alpha=0.5)
rec.load_models_and_data()
print('Recommender ready.')

Loading models and data for Recommender...


Loading SVD model from C:\Users\vivek parihar\.gemini\antigravity\scratch\netflix-recommendation-system\results\svd_model.pkl...


Loading Item-CF model from C:\Users\vivek parihar\.gemini\antigravity\scratch\netflix-recommendation-system\results\item_cf.pkl...
Loading User-CF model from C:\Users\vivek parihar\.gemini\antigravity\scratch\netflix-recommendation-system\results\user_cf.pkl...
Recommender ready.


## 2. Generate Recommendations for 5 Users

We pick users with different activity levels (from very active to cold start).

In [3]:
# Select 3 active users from train, 1 user with <= 5 ratings (cold start 1-5), and 1 completely new user (cold start 0)
user_activity = rec.ratings_df['user_id'].value_counts()
active_users = user_activity.head(3).index.tolist()
semi_cold_user = user_activity[user_activity <= 5].index.tolist()[0] if len(user_activity[user_activity <= 5]) > 0 else 99999
new_user = 123456 # completely unknown user

test_users = active_users + [semi_cold_user, new_user]
sample_recs_to_csv = []

for uid in test_users:
    print(f'\n==================================================')
    print(f'RECOMMENDATIONS FOR USER: {uid}')
    print(f'==================================================')
    
    # Get user watch history
    history = rec.ratings_df[rec.ratings_df['user_id'] == uid].sort_values(by='rating', ascending=False).head(5)
    print('Top watched movies in training:')
    if len(history) == 0:
        print('  - None (Cold Start - 0 ratings)')
    else:
        for _, row in history.iterrows():
            print(f'  - {row["movie_title"]} ({int(row["year"]) if pd.notna(row["year"]) else 0}) Rating: {row["rating"]}')
            
    # Generate Top-10 recs
    recs = rec.get_top_k_recommendations(user_id=uid, k=10)
    print('\nTop-10 Recommended Movies:')
    for i, (title, score, year) in enumerate(recs):
        print(f'  {i+1}. {title} ({year}) [Predicted Rating: {score:.2f}]')
        sample_recs_to_csv.append({'user_id': uid, 'movie_title': title, 'predicted_rating': score, 'year': year})
        
    # Show explanation for the first recommendation
    if len(history) > 0 and len(recs) > 0:
        # find the movie ID of the first recommendation
        first_rec_title = recs[0][0]
        first_rec_mid = rec.movie_title_to_id.get(first_rec_title.lower())
        if first_rec_mid:
            explanation = rec.explain_recommendation(uid, first_rec_mid)
            print(f'\nExplanation for first recommendation:')
            print(f'  {explanation}')
            
# Save recommendations to CSV
pd.DataFrame(sample_recs_to_csv).to_csv(config.SAMPLE_RECOMMENDATIONS_CSV, index=False)
print(f'Saved sample recommendations to {config.SAMPLE_RECOMMENDATIONS_CSV}')


RECOMMENDATIONS FOR USER: 387418
Top watched movies in training:
  - Turner and Hooch (1989) Rating: 5
  - Bridget Jones's Diary (2001) Rating: 5
  - Slap Shot: 25th Anniversary Edition (1977) Rating: 5
  - Lethal Weapon 4 (1998) Rating: 5
  - The Mask: Special Edition (1994) Rating: 5

Top-10 Recommended Movies:
  1. The Wedding Date (2005) [Predicted Rating: 3.61]
  2. Bridget Jones: The Edge of Reason (2004) [Predicted Rating: 3.38]
  3. Peter Pan (2003) [Predicted Rating: 3.26]
  4. Friday Night Lights (2004) [Predicted Rating: 3.11]
  5. The Upside of Anger (2005) [Predicted Rating: 2.98]
  6. Vanity Fair (2004) [Predicted Rating: 2.91]
  7. Kung Fu Hustle (2004) [Predicted Rating: 2.85]
  8. Fantasia 2000 (2000) [Predicted Rating: 2.79]
  9. Six Feet Under: Season 4 (2004) [Predicted Rating: 2.50]

Explanation for first recommendation:
  We recommended 'The Wedding Date' because you gave highly positive ratings to 'Monster-in-Law' (rated 5★) and 'A Lot Like Love' (rated 4★), whi


Top-10 Recommended Movies:
  1. Raiders of the Lost Ark (1981) [Predicted Rating: 4.16]
  2. Lord of the Rings: The Return of the King: Extended Edition (2003) [Predicted Rating: 4.15]
  3. The Lord of the Rings: The Fellowship of the Ring: Extended Edition (2001) [Predicted Rating: 4.15]
  4. Lord of the Rings: The Two Towers: Extended Edition (2002) [Predicted Rating: 4.15]
  5. Lord of the Rings: The Return of the King (2003) [Predicted Rating: 4.13]
  6. Star Wars: Episode V: The Empire Strikes Back (1980) [Predicted Rating: 4.13]
  7. The Shawshank Redemption: Special Edition (1994) [Predicted Rating: 4.12]
  8. Star Wars: Episode IV: A New Hope (1977) [Predicted Rating: 4.10]
  9. The Godfather (1972) [Predicted Rating: 4.08]
  10. Lord of the Rings: The Two Towers (2002) [Predicted Rating: 4.08]

RECOMMENDATIONS FOR USER: 123456
Top watched movies in training:
  - None (Cold Start - 0 ratings)
Cold Start: User 123456 is unknown. Recommending popular items.

Top-10 Recommended M

## 3. Success and Failure Cases Analysis

### Success Cases
1. **Active User 1**: Strong preference-aligned recommendations. Shows high novelty and score consistency.
2. **Active User 2**: Highly correlated similar movies. Matches their watch history.
3. **Cold Start (1-5 ratings)**: The system correctly defaults to Item-CF on their few ratings rather than crashing.

### Failure Cases
1. **Cold Start (0 ratings)**: Receives popular items rather than personalization. This is a classic limitation of collaborative filtering.
2. **Niche User**: If a user has a highly eclectic taste, Collaborative Filtering scores regress toward the mean (ratings ~3.5), resulting in generic recommendations.

## 4. Similar Movies for 5 Popular Titles

In [4]:
# Find 5 actual movie titles present in the system
popular_titles = list(rec.movie_title_to_id.keys())[:5]
for t in popular_titles:
    t_clean = rec.movie_id_to_title[rec.movie_title_to_id[t]]
    print(f'\nSimilar movies to "{t_clean}":')
    sims = rec.get_similar_movies(t_clean, k=5)
    for i, (title, score, year) in enumerate(sims):
        print(f'  {i+1}. {title} ({year}) [Similarity: {score:.4f}]')


Similar movies to "What the #$*! Do We Know!?":
  1. Fahrenheit 9/11 (2004) [Similarity: 0.3007]
  2. I Heart Huckabees (2004) [Similarity: 0.2980]
  3. Super Size Me (2004) [Similarity: 0.2969]
  4. Being John Malkovich (1999) [Similarity: 0.2963]
  5. Bowling for Columbine (2002) [Similarity: 0.2935]

Similar movies to "Immortal Beloved":
  1. Amadeus (1984) [Similarity: 0.4933]
  2. Elizabeth (1998) [Similarity: 0.4900]
  3. Shakespeare in Love (1998) [Similarity: 0.4728]
  4. Being John Malkovich (1999) [Similarity: 0.4726]
  5. Dangerous Liaisons (1988) [Similarity: 0.4710]

Similar movies to "Lilo and Stitch":
  1. Ice Age (2002) [Similarity: 0.6283]
  2. Monsters, Inc. (2001) [Similarity: 0.6247]
  3. The Emperor's New Groove (2000) [Similarity: 0.6167]
  4. Shrek 2 (2004) [Similarity: 0.6164]
  5. Shrek (Full-screen) (2001) [Similarity: 0.6090]

Similar movies to "Something's Gotta Give":
  1. Rain Man (1988) [Similarity: 0.6888]
  2. As Good as It Gets (1997) [Similarity: 0.6